# 02B: Сравнение target — temporal split + эксперимент 09

**Цель:** Сравнение target на temporal split (7 train, 1 val, 2 test). Перебор порогов thr 0.5–0.9, выбор best_thr по avg_net. Oracle как верхняя граница.

**Протокол:** data_labeled + features_selected, LightGBM/CatBoost, signal_flip (HOLD=keep), комиссия 0.1%.

**Target:** tp_sl 2/1, 1/0.5, 0.8/0.4, 1.2/0.6; tb_h* (H=2..7, mult 1.5–2.0) — единая схема tb_h{N}_m{mult}.

## 1. Импорты и загрузка

In [5]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score
from sklearn.dummy import DummyClassifier
import warnings
warnings.filterwarnings('ignore')

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep', '02_targets', '03_features', '04_models', '05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
try:
    import catboost as cb
    HAS_CB = True
except ImportError:
    HAS_CB = False

# Загрузка data_labeled (те же данные, что в 09)
data_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
feat_path = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')
if not os.path.exists(data_path):
    raise FileNotFoundError(f'Не найден: {data_path}. Запустите пайплайн 01-03.')
if not os.path.exists(feat_path):
    raise FileNotFoundError(f'Не найден: {feat_path}')

df = pd.read_parquet(data_path)
with open(feat_path, encoding='utf-8') as f:
    FEATURES = [l.strip() for l in f if l.strip()]
FEATURES = [c for c in FEATURES if c in df.columns]

print(f'Загружено: {len(df):,} строк. Фичей: {len(FEATURES)}')
print(f'LightGBM: {HAS_LGB}, CatBoost: {HAS_CB}')

Загружено: 344,415 строк. Фичей: 22
LightGBM: True, CatBoost: True


## 2. Target: оригинальные + tb_vol из 09

In [6]:
def add_tb_vol(df, by='session_key', vol_col='volatility_14', mult=1.5, h=5, out_col='tb'):
    """Triple-barrier: TP/SL = mult*vol, horizon=h. 1=TP, -1=SL, 0=time."""
    df = df.copy()
    labels = pd.Series(index=df.index, dtype=np.float64)
    for skey, g in df.groupby(by, sort=False):
        idx = g.index
        c = g['close_price'].values
        h_arr, l_arr = g['high'].values, g['low'].values
        v = g[vol_col].fillna(0.01).clip(lower=1e-6).values
        lab = np.full(len(g), np.nan, dtype=np.float64)
        for i in range(len(g) - h):
            up = c[i] * (1 + mult * v[i])
            dn = c[i] * (1 - mult * v[i])
            for j in range(1, h + 1):
                if h_arr[i + j] >= up: lab[i] = 1; break
                if l_arr[i + j] <= dn: lab[i] = -1; break
            else: lab[i] = 0
        labels.loc[idx] = lab
    df[out_col] = labels.values
    return df

def add_target_tp_sl_fixed(df, by='session_key', TP_pct=0.02, SL_pct=0.01, H=20, out_col='tp_sl_fixed'):
    """TP/SL фикс. %: ambiguous -> 0."""
    df = df.copy()
    labels = pd.Series(index=df.index, dtype=np.float64)
    for skey, g in df.groupby(by, sort=False):
        idx, c = g.index, g['close_price'].values
        h_arr, l_arr = g['high'].values, g['low'].values
        lab = np.full(len(g), np.nan, dtype=np.float64)
        for i in range(len(g) - H):
            tp_up, sl_down = c[i] * (1 + TP_pct), c[i] * (1 - SL_pct)
            tp_dn, sl_up = c[i] * (1 - TP_pct), c[i] * (1 + SL_pct)
            res_long = 0
            for j in range(1, H + 1):
                hi, lo = h_arr[i + j], l_arr[i + j]
                if hi >= tp_up and lo <= sl_down: res_long = 0; break
                if hi >= tp_up: res_long = 1; break
                if lo <= sl_down: res_long = -1; break
            res_short = 0
            for j in range(1, H + 1):
                hi, lo = h_arr[i + j], l_arr[i + j]
                if lo <= tp_dn and hi >= sl_up: res_short = 0; break
                if lo <= tp_dn: res_short = 1; break
                if hi >= sl_up: res_short = -1; break
            lab[i] = 1 if (res_long == 1 and res_short != 1) else (-1 if (res_short == 1 and res_long != 1) else 0)
        labels.loc[idx] = lab
    df[out_col] = labels.values
    return df

# tp_sl (фикс. %)
df = add_target_tp_sl_fixed(df, TP_pct=0.02, SL_pct=0.01, H=20, out_col='tp_sl_2_1')
df = add_target_tp_sl_fixed(df, TP_pct=0.01, SL_pct=0.005, H=20, out_col='tp_sl_1_05')
# tp_sl варианты (0.8/0.4, 1.2/0.6)
df = add_target_tp_sl_fixed(df, TP_pct=0.008, SL_pct=0.004, H=20, out_col='tp_sl_0_8_0_4')
df = add_target_tp_sl_fixed(df, TP_pct=0.012, SL_pct=0.006, H=20, out_col='tp_sl_1_2_0_6')
# Топ-4 из 09 + H=2
df = add_tb_vol(df, mult=1.5, h=2, out_col='tb_h2_m1_5')
df = add_tb_vol(df, mult=2.0, h=2, out_col='tb_h2_m2_0')
df = add_tb_vol(df, mult=1.5, h=3, out_col='tb_h3_m1_5')
df = add_tb_vol(df, mult=1.75, h=4, out_col='tb_h4_m1_75')
df = add_tb_vol(df, mult=1.75, h=3, out_col='tb_h3_m1_75')
df = add_tb_vol(df, mult=2.0, h=4, out_col='tb_h4_m2_0')
# tb_vol по H (2..7) — единая схема tb_h{N}_m{mult}
df = add_tb_vol(df, mult=1.5, h=5, out_col='tb_h5_m1_5')
df = add_tb_vol(df, mult=1.75, h=6, out_col='tb_h6_m1_75')
df = add_tb_vol(df, mult=1.75, h=7, out_col='tb_h7_m1_75')

TARGETS_CLASSIC = ['tp_sl_2_1', 'tp_sl_1_05', 'tb_h5_m1_5']  # классика 02
TARGETS_TP_SL = ['tp_sl_2_1', 'tp_sl_1_05', 'tp_sl_0_8_0_4', 'tp_sl_1_2_0_6']
TARGETS_TOP4 = ['tb_h3_m1_5', 'tb_h4_m1_75', 'tb_h3_m1_75', 'tb_h4_m2_0']
TARGETS_H2 = ['tb_h2_m1_5', 'tb_h2_m2_0']
TARGETS_BY_H = ['tb_h5_m1_5', 'tb_h6_m1_75', 'tb_h7_m1_75']
TARGETS = TARGETS_TP_SL + TARGETS_TOP4 + TARGETS_H2 + TARGETS_BY_H

for tcol in TARGETS:
    v = df.dropna(subset=[tcol]).copy()
    v = v[v[tcol].isin([-1.0, 1.0])]
    print(f'{tcol}: BUY={(v[tcol]==1).sum():,} SELL={(v[tcol]==-1).sum():,}')

tp_sl_2_1: BUY=51,120 SELL=44,609
tp_sl_1_05: BUY=91,983 SELL=89,820
tp_sl_0_8_0_4: BUY=100,988 SELL=99,999
tp_sl_1_2_0_6: BUY=82,911 SELL=78,910
tb_h3_m1_5: BUY=103,950 SELL=102,030
tb_h4_m1_75: BUY=104,879 SELL=103,305
tb_h3_m1_75: BUY=87,772 SELL=86,255
tb_h4_m2_0: BUY=90,939 SELL=89,060
tb_h2_m1_5: BUY=80,064 SELL=78,091
tb_h2_m2_0: BUY=52,237 SELL=50,373
tb_h5_m1_5: BUY=131,193 SELL=129,458
tb_h6_m1_75: BUY=126,696 SELL=125,506
tb_h7_m1_75: BUY=133,805 SELL=132,623


## 3. Temporal split

In [7]:
df['ret_next'] = df.groupby('session_key')['close_price'].pct_change().shift(-1)
df['date'] = pd.to_datetime(df['datetime'], utc=True).dt.date
dates = sorted(df['date'].unique())

# 7 train, 1 val, 2 test (как в 09)
train_dates = set(dates[:7])
val_dates = set([dates[7]])
test_dates = set(dates[8:10])

print(f'Train: {min(train_dates)}..{max(train_dates)}, Val: {dates[7]}, Test: {dates[8]}, {dates[9]}')

Train: 2026-02-01..2026-02-07, Val: 2026-02-08, Test: 2026-02-09, 2026-02-10


## 4. Обучение моделей и бектест

In [8]:
THR_LIST = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
N_TEST_DAYS = 2

def _run_pnl(proba, bt_df, thr=0.6, commission=0.001):
    """signal_flip: BUY>=thr, SELL<=1-thr, HOLD иначе (держим позицию)."""
    thresh_hi, thresh_lo = thr, 1 - thr
    signal = np.where(proba >= thresh_hi, 1, np.where(proba <= thresh_lo, -1, 0))
    ret = bt_df['ret_next'].values
    sess = bt_df['session_key'].values
    pos = np.zeros(len(signal), dtype=float)
    prev = 0.0
    for i in range(len(signal)):
        if i and sess[i] != sess[i-1]: prev = 0.0
        pos[i] = signal[i] if signal[i] != 0 else prev
        if signal[i] != 0: prev = signal[i]
    last = np.zeros(len(signal), dtype=bool)
    last[:-1] = sess[1:] != sess[:-1]
    last[-1] = True
    pos = np.where(last & (pos != 0), 0.0, pos)
    pos_prev = np.roll(pos, 1); pos_prev[0] = 0.0
    chg = np.zeros(len(pos), dtype=bool); chg[1:] = sess[1:] != sess[:-1]
    pos_prev = np.where(chg, 0.0, pos_prev)
    nch = int(((pos != pos_prev) & ((pos != 0) | (pos_prev != 0))).sum())
    gross = (pos * ret).sum() * 100
    net = gross - nch * (commission / 2) * 100
    return net, nch, net / nch if nch else np.nan

def eval_metrics(proba, y_true):
    pred = (proba >= 0.5).astype(int)
    return {'AUC': roc_auc_score(y_true, proba) if len(np.unique(y_true)) > 1 else 0.5,
            'F1': f1_score(y_true, pred, zero_division=0)}

COMMISSION = 0.001
all_metrics, all_pnl = [], []

def _best_thr_sweep(proba, bt_df):
    best_avg, best_thr, best_net, best_nc = np.nan, None, np.nan, 0
    for thr in THR_LIST:
        net, nc, avg = _run_pnl(proba, bt_df, thr=thr, commission=COMMISSION)
        if not np.isnan(avg) and (np.isnan(best_avg) or avg > best_avg):
            best_avg, best_thr, best_net, best_nc = avg, thr, net, nc
    return best_thr, best_net, best_nc, best_avg

for TCOL in TARGETS:
    valid = df.dropna(subset=[TCOL] + FEATURES + ['ret_next']).copy()
    valid = valid[valid[TCOL].isin([-1.0, 1.0])]
    valid['y'] = (valid[TCOL] == 1).astype(int)
    train_df = valid[valid['date'].isin(train_dates)]
    if len(train_df) < 500:
        print(f'Пропуск {TCOL}: train={len(train_df)}')
        continue
    valid_all = df.dropna(subset=[TCOL] + FEATURES + ['ret_next']).copy()
    valid_all = valid_all[valid_all[TCOL].isin([-1.0, 0.0, 1.0])]
    bt = valid_all[valid_all['date'].isin(test_dates)].dropna(subset=['ret_next'])
    if len(bt) < 50:
        print(f'Пропуск {TCOL}: test bars={len(bt)}')
        continue
    test_df = valid[valid['date'].isin(test_dates)]
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[FEATURES].fillna(0))
    X_test = scaler.transform(test_df[FEATURES].fillna(0))
    y_train, y_test = train_df['y'].values, test_df['y'].values

    X_bt = scaler.transform(bt[FEATURES].fillna(0))

    # Dummy
    dummy = DummyClassifier(strategy='stratified', random_state=42).fit(X_train, y_train)
    all_metrics.append(dict(eval_metrics(dummy.predict_proba(X_test)[:, 1], y_test), target=TCOL, model='Dummy'))
    thr_d, net, nc, avg = _best_thr_sweep(dummy.predict_proba(X_bt)[:, 1], bt)
    all_pnl.append({'target': TCOL, 'model': 'Dummy', 'best_thr': thr_d, 'net_%': net, 'trades': nc, 'trades_per_day': nc / N_TEST_DAYS, 'avg_net': avg})

    if HAS_LGB:
        m = lgb.LGBMClassifier(n_estimators=200, max_depth=8, learning_rate=0.05, random_state=42, verbose=-1)
        m.fit(X_train, y_train)
        all_metrics.append(dict(eval_metrics(m.predict_proba(X_test)[:, 1], y_test), target=TCOL, model='LightGBM'))
        thr_l, net, nc, avg = _best_thr_sweep(m.predict_proba(X_bt)[:, 1], bt)
        all_pnl.append({'target': TCOL, 'model': 'LightGBM', 'best_thr': thr_l, 'net_%': net, 'trades': nc, 'trades_per_day': nc / N_TEST_DAYS, 'avg_net': avg})
    if HAS_CB:
        m = cb.CatBoostClassifier(iterations=200, depth=8, learning_rate=0.05, random_state=42, verbose=0)
        m.fit(X_train, y_train)
        all_metrics.append(dict(eval_metrics(m.predict_proba(X_test)[:, 1], y_test), target=TCOL, model='CatBoost'))
        thr_c, net, nc, avg = _best_thr_sweep(m.predict_proba(X_bt)[:, 1], bt)
        all_pnl.append({'target': TCOL, 'model': 'CatBoost', 'best_thr': thr_c, 'net_%': net, 'trades': nc, 'trades_per_day': nc / N_TEST_DAYS, 'avg_net': avg})

res_df = pd.DataFrame(all_metrics)
pnl_df = pd.DataFrame(all_pnl)
print('Метрики (test), AUC/F1:')
print(res_df.pivot_table(index=['target', 'model'], values=['AUC', 'F1']).round(4).to_string())

Метрики (test), AUC/F1:
                           AUC      F1
target        model                   
tb_h2_m1_5    CatBoost  0.7260  0.6771
              Dummy     0.4998  0.5072
              LightGBM  0.7365  0.6823
tb_h2_m2_0    CatBoost  0.7570  0.7037
              Dummy     0.4983  0.5100
              LightGBM  0.7679  0.7127
tb_h3_m1_5    CatBoost  0.7123  0.6654
              Dummy     0.5024  0.5086
              LightGBM  0.7190  0.6670
tb_h3_m1_75   CatBoost  0.7287  0.6786
              Dummy     0.5000  0.5064
              LightGBM  0.7364  0.6807
tb_h4_m1_75   CatBoost  0.7157  0.6650
              Dummy     0.4979  0.5042
              LightGBM  0.7242  0.6693
tb_h4_m2_0    CatBoost  0.7266  0.6765
              Dummy     0.4969  0.5043
              LightGBM  0.7347  0.6811
tb_h5_m1_5    CatBoost  0.6919  0.6434
              Dummy     0.5043  0.5090
              LightGBM  0.6966  0.6460
tb_h6_m1_75   CatBoost  0.6918  0.6409
              Dummy     0.4978  0.5016
 

In [9]:
# Oracle: идеальный бектест по target (торгуем 1=long, -1=short напрямую)
oracle_rows = []
for TCOL in TARGETS:
    valid = df.dropna(subset=[TCOL, 'ret_next']).copy()
    valid = valid[valid[TCOL].isin([-1.0, 1.0])]
    bt = valid[valid['date'].isin(test_dates)].dropna(subset=['ret_next'])
    if len(bt) < 10:
        continue
    pos = bt[TCOL].values.astype(float)
    ret = bt['ret_next'].values
    sess = bt['session_key'].values
    pos_prev = np.roll(pos, 1); pos_prev[0] = 0.0
    chg = np.zeros(len(pos), dtype=bool); chg[1:] = sess[1:] != sess[:-1]
    pos_prev = np.where(chg, 0.0, pos_prev)
    nch = int(((pos != pos_prev) & ((pos != 0) | (pos_prev != 0))).sum())
    net = (pos * ret).sum() * 100 - nch * (COMMISSION / 2) * 100
    oracle_rows.append({'target': TCOL, 'oracle_net_%': net, 'oracle_trades': nch, 'oracle_trades_per_day': nch / N_TEST_DAYS})
oracle_df = pd.DataFrame(oracle_rows)

## 5. Сводка PnL — tp_sl + tb_h5 (классика)

In [10]:
print('--- Оригинальные target ---')
orig = pnl_df[pnl_df['target'].isin(TARGETS_CLASSIC)]
print(orig[['target', 'model', 'best_thr', 'net_%', 'trades', 'trades_per_day', 'avg_net']].to_string(index=False))

--- Оригинальные target ---
    target    model  best_thr        net_%  trades  trades_per_day   avg_net
 tp_sl_2_1    Dummy      0.50 -1731.527861   35391         17695.5 -0.048926
 tp_sl_2_1 LightGBM      0.90    40.804219      26            13.0  1.569393
 tp_sl_2_1 CatBoost      0.90    87.986877      44            22.0  1.999702
tp_sl_1_05    Dummy      0.50 -1851.737064   35504         17752.0 -0.052156
tp_sl_1_05 LightGBM      0.90    36.187832      26            13.0  1.391840
tp_sl_1_05 CatBoost      0.90    98.848753      62            31.0  1.594335
tb_h5_m1_5    Dummy      0.50 -1947.360129   37028         18514.0 -0.052592
tb_h5_m1_5 LightGBM      0.85   199.704826     128            64.0  1.560194
tb_h5_m1_5 CatBoost      0.85   344.163891     197            98.5  1.747025


## 6. Сводка PnL — топ-4 из 09 + H2 + tp_sl варианты

In [11]:
print('--- Топ-4 из 09 + H2 + tp_sl 0.8/0.4, 1.2/0.6 ---')
sub = pnl_df[pnl_df['target'].isin(TARGETS_TOP4 + TARGETS_H2 + TARGETS_TP_SL)]
print(sub[['target', 'model', 'best_thr', 'net_%', 'trades', 'trades_per_day', 'avg_net']].to_string(index=False))

--- Топ-4 из 09 + H2 + tp_sl 0.8/0.4, 1.2/0.6 ---
       target    model  best_thr        net_%  trades  trades_per_day   avg_net
    tp_sl_2_1    Dummy      0.50 -1731.527861   35391         17695.5 -0.048926
    tp_sl_2_1 LightGBM      0.90    40.804219      26            13.0  1.569393
    tp_sl_2_1 CatBoost      0.90    87.986877      44            22.0  1.999702
   tp_sl_1_05    Dummy      0.50 -1851.737064   35504         17752.0 -0.052156
   tp_sl_1_05 LightGBM      0.90    36.187832      26            13.0  1.391840
   tp_sl_1_05 CatBoost      0.90    98.848753      62            31.0  1.594335
tp_sl_0_8_0_4    Dummy      0.50 -1851.109270   35536         17768.0 -0.052091
tp_sl_0_8_0_4 LightGBM      0.85   301.909221     243           121.5  1.242425
tp_sl_0_8_0_4 CatBoost      0.75  1258.791468     935           467.5  1.346301
tp_sl_1_2_0_6    Dummy      0.50 -1719.227028   35518         17759.0 -0.048404
tp_sl_1_2_0_6 LightGBM      0.80   807.733872     618           309.0 

## 7. Сводка PnL — по H (6, 7)

In [12]:
print('--- tb_vol по H: H5, H6, H7 ---')
byh = pnl_df[pnl_df['target'].isin(TARGETS_BY_H)]
print(byh[['target', 'model', 'best_thr', 'net_%', 'trades', 'trades_per_day', 'avg_net']].to_string(index=False))

--- tb_vol по H: H5, H6, H7 ---
     target    model  best_thr        net_%  trades  trades_per_day   avg_net
 tb_h5_m1_5    Dummy      0.50 -1947.360129   37028         18514.0 -0.052592
 tb_h5_m1_5 LightGBM      0.85   199.704826     128            64.0  1.560194
 tb_h5_m1_5 CatBoost      0.85   344.163891     197            98.5  1.747025
tb_h6_m1_75    Dummy      0.50 -1998.465821   36944         18472.0 -0.054094
tb_h6_m1_75 LightGBM      0.80   617.785097     463           231.5  1.334309
tb_h6_m1_75 CatBoost      0.80   789.038120     468           234.0  1.685979
tb_h7_m1_75    Dummy      0.50 -2050.414565   36847         18423.5 -0.055647
tb_h7_m1_75 LightGBM      0.75  1347.769621     899           449.5  1.499188
tb_h7_m1_75 CatBoost      0.85   306.498644     161            80.5  1.903718


## 8. Oracle (верхняя граница профита по target)

In [13]:
print('Oracle: идеальный бектест по target на test')
print(oracle_df.to_string(index=False))

Oracle: идеальный бектест по target на test
       target  oracle_net_%  oracle_trades  oracle_trades_per_day
    tp_sl_2_1   4856.833367           1600                  800.0
   tp_sl_1_05   8995.502240           4986                 2493.0
tp_sl_0_8_0_4  10115.919392           6548                 3274.0
tp_sl_1_2_0_6   7946.198633           3809                 1904.5
   tb_h3_m1_5  11481.792698          13179                 6589.5
  tb_h4_m1_75  10656.842501          11290                 5645.0
  tb_h3_m1_75  10363.881928          10648                 5324.0
   tb_h4_m2_0   9635.422419           9351                 4675.5
   tb_h2_m1_5  10932.684640          12154                 6077.0
   tb_h2_m2_0   8369.768495           7600                 3800.0
   tb_h5_m1_5  11811.873701          14339                 7169.5
  tb_h6_m1_75  10901.581445          12214                 6107.0
  tb_h7_m1_75  10947.003197          12592                 6296.0


## 8. Сравнение: LightGBM по всем target

In [14]:
lgb_pnl = pnl_df[(pnl_df['model'] == 'LightGBM')].sort_values('avg_net', ascending=False)
merged = lgb_pnl.merge(oracle_df, on='target', how='left')
print('LightGBM vs Oracle (отсортировано по avg_net):')
print(merged[['target', 'best_thr', 'net_%', 'trades', 'trades_per_day', 'avg_net', 'oracle_net_%', 'oracle_trades_per_day']].to_string(index=False))

LightGBM vs Oracle (отсортировано по avg_net):
       target  best_thr       net_%  trades  trades_per_day  avg_net  oracle_net_%  oracle_trades_per_day
   tb_h2_m2_0      0.90  150.555227      85            42.5 1.771238   8369.768495                 3800.0
    tp_sl_2_1      0.90   40.804219      26            13.0 1.569393   4856.833367                  800.0
   tb_h5_m1_5      0.85  199.704826     128            64.0 1.560194  11811.873701                 7169.5
   tb_h2_m1_5      0.85  486.142626     316           158.0 1.538426  10932.684640                 6077.0
  tb_h4_m1_75      0.80 1019.765743     676           338.0 1.508529  10656.842501                 5645.0
  tb_h7_m1_75      0.75 1347.769621     899           449.5 1.499188  10947.003197                 6296.0
   tp_sl_1_05      0.90   36.187832      26            13.0 1.391840   8995.502240                 2493.0
  tb_h3_m1_75      0.80 1039.531482     757           378.5 1.373225  10363.881928                 5324.0

## 10. Выводы и рекомендации

**Протокол:** temporal split (7 train, 1 val, 2 test), перебор thr 0.5–0.9, best_thr по avg_net. Oracle — верхняя граница.

---

### AUC и предсказуемость
- **tb_h2** (H=2) — самый высокий AUC: 0.72–0.77 (tb_h2_m2_0 ≈ 0.77).
- При росте H (5→7) AUC падает: ~0.77 → ~0.69. Target с длинным горизонтом хуже предсказывается.
- **tp_sl** — AUC 0.66–0.69, ниже tb_h.
- Dummy ≈ 0.50 везде — сигнал не случайный.

### PnL и avg_net
- **Лучший avg_net (LightGBM):** tb_h2_m2_0 (1.77), tb_h5_m1_5 (1.56), tb_h2_m1_5 (1.54).
- **Максимальный net_%:** tb_h7_m1_75 — LightGBM 1347.8% (899 trades, avg_net 1.50).
- **Классика (tp_sl, tb_h5):** CatBoost лучше по avg_net; tb_h5 даёт 199–344% net vs 37–99% у tp_sl.

### tp_sl vs tb_vol
- **tb_vol** заметно лучше по AUC и PnL. tp_sl — мало сделок (26–62), консервативные пороги (0.85–0.90).
- tb_vol H=2..7 даёт больше trades и выше net_% при лучшей предсказуемости.

### Горизонт H (tb_vol)
- H=2: лучший AUC и avg_net — самый «чистый» target.
- H=3–4: AUC 0.71–0.73, высокая net_% (771–1039% LightGBM).
- H=5–7: AUC ниже, но net_% растёт за счёт числа сделок.

### Рекомендации для prod
1. **Качество сделок (avg_net):** tb_h2_m2_0 или tb_h2_m1_5 — CatBoost/LightGBM.
2. **Максимум net_%:** tb_h7_m1_75 (LightGBM) или tb_h6_m1_75 (CatBoost), tb_h4_m1_75 — стабильный вариант.
3. **Пороги:** 0.80–0.90; tp_sl — 0.85–0.90, tb_h6–7 — 0.75–0.80.
4. **Внимание:** только 2 дня теста — нужен forward test на новых периодах.